# **1. Coleta dos dados**

Este notebook coleta as partidas do Campeonato Brasileiro Série A entre 2020 e 2026. Cada partida é identificada pelo `game_id` e vinculada aos metadados, registros disciplinares e documentos publicados pela CBF.

Os arquivos JSON e PDF são armazenados de forma incremental para permitir auditoria e reexecução.

In [1]:
from __future__ import annotations

import gzip
import json
import os
import re
import subprocess
import time
import threading
import unicodedata
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import requests
import urllib3
from bs4 import BeautifulSoup
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

RAW = ROOT / "data" / "raw"
INTERIM = ROOT / "data" / "interim"
PROCESSED = ROOT / "data" / "processed"
CACHE = INTERIM / "game_api_cache"
PDF_DIR = RAW / "match-reports"
for directory in (RAW, INTERIM, PROCESSED, CACHE, PDF_DIR):
    directory.mkdir(parents=True, exist_ok=True)

API_GAME = "https://www.cbf.com.br/api/cbf/jogos/{game_id}"
HEADERS = {"User-Agent": "academic-research/cbf-match-reports-nlp"}
COMPLETED_YEARS = list(range(2020, 2026))
CURRENT_SEASON = 2026
YEARS = COMPLETED_YEARS + [CURRENT_SEASON]
MAX_WORKERS = 2
REQUEST_TIMEOUT = 45
REQUEST_INTERVAL = 0.35
BATCH_SIZE = 60
BATCH_PAUSE = 6
VERIFY_CBF_SSL = os.getenv("CBF_VERIFY_SSL", "0") == "1"
if not VERIFY_CBF_SSL:
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
DOWNLOAD_PDFS = True
REFRESH_API = False

## **2. Seleção das partidas**

A base contém 380 partidas por temporada entre 2020 e 2025. A temporada de 2026 é atualizada conforme as partidas são disponibilizadas pela CBF. Os identificadores são obtidos nas páginas oficiais e nos endpoints das 38 rodadas.

In [2]:
def strip_accents(value: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", str(value)) if not unicodedata.combining(c))


def discovery_request(url: str, as_json: bool = False):
    last_error = None
    for attempt in range(5):
        try:
            time.sleep(REQUEST_INTERVAL)
            result = subprocess.run(
                ["curl", "-sSL", "--fail", "--retry", "2", "--retry-all-errors", "--connect-timeout", "15", "--max-time", str(REQUEST_TIMEOUT), "-A", HEADERS["User-Agent"], url],
                check=True, capture_output=True,
            )
            if not result.stdout:
                raise ValueError("resposta vazia")
            text = result.stdout.decode("utf-8")
            return json.loads(text) if as_json else text
        except Exception as exc:
            last_error = exc
            if attempt < 4:
                time.sleep(min(180, 30 * (attempt + 1)))
    raise RuntimeError(f"CBF indisponível após recuo: {last_error!r}")


def discover_game_ids(years=YEARS) -> list[int]:
    ids: set[int] = set()
    for year in tqdm(years, desc="Descobrindo temporadas"):
        url = f"https://www.cbf.com.br/futebol-brasileiro/tabelas/campeonato-brasileiro/serie-a/{year}"
        try:
            html = discovery_request(url)
            competition_match = re.search(r'competitionId\\":\\"(\d+)', html)
            phase_match = re.search(r'fase_id\\":\\"(\d+)', html)
            if not competition_match or not phase_match:
                raise ValueError(f"competitionId/fase_id não encontrados para {year}")
            competition_id = competition_match.group(1)
            phase_id = phase_match.group(1)
            for round_number in range(1, 39):
                endpoint = f"https://www.cbf.com.br/api/cbf/jogos/campeonato/{competition_id}/rodada/{round_number}/fase/{phase_id}"
                payload = discovery_request(endpoint, as_json=True)
                stack = [payload]
                while stack:
                    item = stack.pop()
                    if isinstance(item, dict):
                        if item.get("id_jogo"):
                            game_id = int(item["id_jogo"])
                            ids.add(game_id)
                            item.setdefault("ano", year)
                            item.setdefault("campeonato", {"id": competition_id, "ano": year, "nome": "Campeonato Brasileiro Série A", "nome_categoria": "Masculino"})
                            cache_path = CACHE / f"{game_id}.json.gz"
                            with gzip.open(cache_path, "wt", encoding="utf-8") as handle:
                                json.dump({"jogo": item}, handle, ensure_ascii=False)
                        stack.extend(item.values())
                    elif isinstance(item, list):
                        stack.extend(item)
        except Exception as exc:
            print(f"Falha na descoberta de {year}: {exc!r}")
    return sorted(ids)


cached_by_year = {year: [] for year in YEARS}
for path in CACHE.glob("*.json.gz"):
    try:
        with gzip.open(path, "rt", encoding="utf-8") as handle:
            cached_game = json.load(handle).get("jogo", {})
        cached_year = int(cached_game.get("ano"))
        if cached_year in cached_by_year:
            cached_by_year[cached_year].append(int(cached_game.get("id_jogo") or path.name.split(".")[0]))
    except Exception:
        continue

years_to_discover = [year for year in COMPLETED_YEARS if len(set(cached_by_year[year])) != 380]
years_to_discover.append(CURRENT_SEASON)
discovered_ids = discover_game_ids(years_to_discover)
cached_ids = [game_id for ids in cached_by_year.values() for game_id in ids]
game_ids = sorted(set(cached_ids).union(discovered_ids))
universe_source = "Brasileirão Série A 2020–2026, CBF e cache local"

print(f"{len(game_ids):,} partidas | fonte: {universe_source}")
assert 2_280 < len(game_ids) <= 2_660, f"Quantidade inesperada de partidas: {len(game_ids)}"

Descobrindo temporadas:   0%|          | 0/1 [00:00<?, ?it/s]

2,660 partidas | fonte: Brasileirão Série A 2020–2026, CBF e cache local


In [3]:
_request_lock = threading.Lock()
_last_request_at = 0.0


def controlled_get(url: str):
    global _last_request_at
    with _request_lock:
        wait = REQUEST_INTERVAL - (time.monotonic() - _last_request_at)
        if wait > 0:
            time.sleep(wait)
        _last_request_at = time.monotonic()
    return requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT, verify=VERIFY_CBF_SSL)


def fetch_game(game_id: int, refresh: bool = False) -> dict:
    cache_path = CACHE / f"{game_id}.json.gz"
    if cache_path.exists() and not refresh:
        with gzip.open(cache_path, "rt", encoding="utf-8") as handle:
            return json.load(handle)

    last_error = None
    for attempt in range(4):
        try:
            response = controlled_get(API_GAME.format(game_id=game_id))
            response.raise_for_status()
            payload = response.json()
            if not payload.get("jogo"):
                raise ValueError("Resposta sem objeto 'jogo'")
            with gzip.open(cache_path, "wt", encoding="utf-8") as handle:
                json.dump(payload, handle, ensure_ascii=False)
            return payload
        except Exception as exc:
            last_error = exc
            time.sleep(5 * (attempt + 1))
    return {"_error": repr(last_error), "_game_id": game_id}


payloads = {}
progress = tqdm(total=len(game_ids), desc="Partidas CBF")
for batch_start in range(0, len(game_ids), BATCH_SIZE):
    batch = game_ids[batch_start:batch_start + BATCH_SIZE]
    batch_requires_network = REFRESH_API or any(not (CACHE / f"{game_id}.json.gz").exists() for game_id in batch)
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(fetch_game, game_id, REFRESH_API): game_id for game_id in batch}
        for future in as_completed(futures):
            game_id = futures[future]
            try:
                payloads[game_id] = future.result()
            except Exception as exc:
                payloads[game_id] = {"_error": repr(exc), "_game_id": game_id}
            progress.update(1)
    if batch_requires_network and batch_start + BATCH_SIZE < len(game_ids):
        time.sleep(BATCH_PAUSE)
progress.close()

failed = {key: value for key, value in payloads.items() if not value.get("jogo")}
print(f"sucesso={len(payloads)-len(failed):,} | falhas={len(failed):,}")

Partidas CBF:   0%|          | 0/2660 [00:00<?, ?it/s]

sucesso=2,660 | falhas=0


## **3. Estruturação dos dados**

`games.csv` contém uma linha por partida e diferencia jogos realizados, jogos futuros, datas a definir e súmulas pendentes. `disciplinary_records.csv` reúne os registros disciplinares estruturados, enquanto o índice de súmulas relaciona cada `game_id` ao PDF correspondente.

In [4]:
def first_dict(value):
    if isinstance(value, dict):
        return value
    if isinstance(value, list) and value:
        return value[0]
    return {}


def team_name(value):
    obj = first_dict(value)
    return obj.get("nome") or obj.get("nome_popular") or obj.get("nome_clube")


def extract_referee(arbitros):
    for item in arbitros or []:
        role = strip_accents(item.get("funcao", "")).casefold().strip()
        if role in {"arbitro", "arbitro principal"}:
            return item.get("nome")
    return None


def sumula_url(documentos):
    for item in documentos or []:
        title = strip_accents(item.get("title") or item.get("titulo") or item.get("nome") or "").casefold()
        if "sumula" in title:
            return item.get("url") or item.get("arquivo")
    return None


games_rows, records_rows = [], []
for requested_id, payload in payloads.items():
    jogo = payload.get("jogo")
    if not jogo:
        continue
    competition_raw = jogo.get("campeonato")
    competition = first_dict(competition_raw)
    competition_name = competition.get("nome") or (competition_raw if isinstance(competition_raw, str) else None)
    game_id = int(jogo.get("id_jogo") or requested_id)
    home = team_name(jogo.get("mandante"))
    away = team_name(jogo.get("visitante"))
    games_rows.append({
        "game_id": game_id,
        "match_date": jogo.get("data") or jogo.get("data_realizacao"),
        "year": competition.get("ano") or jogo.get("ano"),
        "competition_id": competition.get("id"),
        "competition": competition_name,
        "category": competition.get("nome_categoria") or "Série A",
        "round": jogo.get("rodada"),
        "home_team": home,
        "away_team": away,
        "referee": extract_referee(jogo.get("arbitros")),
        "sumula_url": sumula_url(jogo.get("documentos")),
    })
    for record in (jogo.get("registros") or jogo.get("penalidades") or []):
        if strip_accents(record.get("tipo", "")).upper() != "PENALIDADE":
            continue
        records_rows.append({
            "game_id": game_id,
            "record_id": record.get("id"),
            "result": record.get("resultado"),
            "team": record.get("clube_nome") or record.get("clube"),
            "team_id": record.get("id_clube") or record.get("clube_id"),
            "player": record.get("atleta_nome") or record.get("atleta_apelido"),
            "player_nickname": record.get("atleta_apelido"),
            "player_id": record.get("id_atleta") or record.get("atleta_id"),
            "shirt_number": record.get("numero_camisa") or record.get("atleta_camisa"),
            "period": record.get("periodo") or record.get("tempo_jogo"),
            "minute": int(re.match(r"\d+", str(record.get("minuto") or record.get("minutos") or 0)).group()),
        })

games = pd.DataFrame(games_rows).drop_duplicates("game_id").sort_values(["year", "game_id"])
parsed_match_dates = pd.to_datetime(games["match_date"].astype("string").str.strip(), format="%d/%m/%Y", errors="coerce")
today = pd.Timestamp.now(tz="America/Sao_Paulo").normalize().tz_localize(None)
games["match_status"] = "report_pending"
games.loc[games["sumula_url"].notna(), "match_status"] = "played_with_report"
games.loc[games["sumula_url"].isna() & parsed_match_dates.ge(today), "match_status"] = "future_scheduled"
games.loc[games["sumula_url"].isna() & parsed_match_dates.isna(), "match_status"] = "date_to_be_defined"
games["referee_status"] = "available"
games.loc[games["referee"].isna() & games["sumula_url"].notna(), "referee_status"] = "missing_with_report"
games.loc[games["referee"].isna() & games["match_status"].isin(["future_scheduled", "date_to_be_defined"]), "referee_status"] = "pending_future_match"
games.loc[games["referee"].isna() & games["match_status"].eq("report_pending"), "referee_status"] = "missing_report_pending"
discipline = pd.DataFrame(records_rows)
games.to_csv(INTERIM / "games.csv", index=False)
discipline.to_csv(INTERIM / "disciplinary_records.csv", index=False)
display(games.head(3))
display(games.groupby(["year", "competition"], dropna=False).size().rename("matches").to_frame())

,game_id,match_date,year,competition_id,competition,category,round,home_team,away_team,referee,sumula_url,match_status,referee_status
0,815072,09/08/2020,2020,None,Campeonato Brasileiro,Série A,1,Flamengo,Atlético Mineiro,Raphael Claus,https://conteudo.cbf.com.br/sumulas/2020/1421s...,played_with_report,available
1,815073,30/09/2020,2020,None,Campeonato Brasileiro,Série A,1,Botafogo de Futebol E Regatas,Esporte Clube Bahia,Savio Pereira Sampaio,https://conteudo.cbf.com.br/sumulas/2020/1422s...,played_with_report,available
2,815074,26/01/2021,2020,None,Campeonato Brasileiro,Série A,1,Palmeiras,Vasco da Gama,Anderson Daronco,https://conteudo.cbf.com.br/sumulas/2020/1423s...,played_with_report,available


,,matches
year,competition,
2020,Campeonato Brasileiro,380
2021,Campeonato Brasileiro,380
2022,Campeonato Brasileiro,380
2023,Campeonato Brasileiro,380
2024,Campeonato Brasileiro,380
2025,Campeonato Brasileiro,380
2026,Campeonato Brasileiro,380


In [5]:
def safe_slug(value):
    value = strip_accents(value or "unknown").casefold()
    return re.sub(r"[^a-z0-9]+", "-", value).strip("-")


def download_pdf(row) -> dict:
    url = row.sumula_url
    if not isinstance(url, str) or not url.startswith("http"):
        return {"game_id": row.game_id, "pdf_path": None, "download_status": "missing_url"}
    folder = PDF_DIR / str(int(row.year)) / safe_slug(row.competition)
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / f"{int(row.game_id)}.pdf"
    if path.exists() and path.stat().st_size > 1_000:
        return {"game_id": row.game_id, "pdf_path": str(path.relative_to(ROOT)), "download_status": "cached"}
    try:
        response = controlled_get(url)
        response.raise_for_status()
        if not response.content.startswith(b"%PDF"):
            raise ValueError("conteúdo não é PDF")
        path.write_bytes(response.content)
        return {"game_id": row.game_id, "pdf_path": str(path.relative_to(ROOT)), "download_status": "downloaded"}
    except Exception as exc:
        return {"game_id": row.game_id, "pdf_path": None, "download_status": f"error:{type(exc).__name__}"}


if DOWNLOAD_PDFS:
    downloads = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(download_pdf, row) for row in games.itertuples(index=False)]
        for future in tqdm(as_completed(futures), total=len(futures), desc="Súmulas"):
            downloads.append(future.result())
    pdf_status = pd.DataFrame(downloads)
else:
    pdf_status = games[["game_id"]].assign(pdf_path=None, download_status="disabled")

index = games.merge(pdf_status, on="game_id", how="left")
index.to_csv(PDF_DIR / "match_reports_index.csv", index=False)
display(index["download_status"].value_counts(dropna=False).to_frame("matches"))

Súmulas:   0%|          | 0/2660 [00:00<?, ?it/s]

,matches
download_status,
cached,2485
missing_url,165
downloaded,10


In [6]:
audit = pd.Series({
    "seed_game_ids": len(game_ids),
    "api_games": games.game_id.nunique(),
    "games_with_sumula_url": games.sumula_url.notna().sum(),
    "local_pdfs": index.pdf_path.notna().sum(),
    "disciplinary_records": len(discipline),
    "future_scheduled_games": games.match_status.eq("future_scheduled").sum(),
    "games_with_date_to_be_defined": games.match_status.eq("date_to_be_defined").sum(),
    "past_games_with_report_pending": games.match_status.eq("report_pending").sum(),
    "referee_pending_future_match": games.referee_status.eq("pending_future_match").sum(),
    "missing_referee_with_report": games.referee_status.eq("missing_with_report").sum(),
    "missing_referee_report_pending": games.referee_status.eq("missing_report_pending").sum(),
}, name="value")
display(audit.to_frame())
display(pd.crosstab(games["match_status"], games["referee_status"], margins=True))
assert games.year.isin(YEARS).all(), "A coleta contém temporada fora do recorte"
assert games.competition.str.contains("Brasileiro", case=False, na=False).all(), "A coleta contém outra competição"
season_counts = games.groupby("year").game_id.nunique().reindex(YEARS, fill_value=0)
assert season_counts.reindex(COMPLETED_YEARS).eq(380).all(), f"Temporadas completas com partidas ausentes: {season_counts.to_dict()}"
assert 0 < season_counts.loc[CURRENT_SEASON] <= 380, f"Quantidade inválida para 2026: {season_counts.loc[CURRENT_SEASON]}"
assert games.game_id.nunique() == int(season_counts.sum()), "Há inconsistência na contagem de partidas"

,value
seed_game_ids,2660
api_games,2660
games_with_sumula_url,2495
local_pdfs,2495
disciplinary_records,13762
future_scheduled_games,20
games_with_date_to_be_defined,145
past_games_with_report_pending,0
referee_pending_future_match,155
missing_referee_with_report,0


referee_status,available,pending_future_match,All
match_status,,,
date_to_be_defined,0,145,145
future_scheduled,10,10,20
played_with_report,2495,0,2495
All,2505,155,2660
